In [1]:
# ==============================================================
# 纯净版消融实验：分类 (LDA) 与回归 (Random Forest)
# 综合对比测试：7 种 SOTA MOEAs (包含 MOFS-REPLS 和 DREA-FS 专版)
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy openpyxl
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random
import time
from scipy import stats

# --- Sklearn ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.metrics import r2_score, accuracy_score
from sklearn.cluster import KMeans

# --- Models ---
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, RandomForestRegressor 
from sklearn.svm import SVC, SVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression

# --- Pymoo (v0.6.0+) ---
from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.algorithms.moo.rvea import RVEA
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排列顺序
METHOD_ORDER = ["NSGA-II", "NSGA-III", "RVEA", "SPEA2", "MOEA/D", "MOFS-REPLS", "DREA-FS"]

# ==============================================================
# 0. Configuration
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

# ==============================================================
# 1. Safe & Fast Fitness Functions (统一目标空间)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.02):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

class DualTaskFeatureSelection(Problem):
    def __init__(self, opts, **kwargs):
        # 3 个目标：1. 最大化分类重要性, 2. 最小化冗余, 3. 最大化回归R2
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1 = np.zeros(X.shape[0])
        f2 = np.zeros(X.shape[0])
        f3 = np.zeros(X.shape[0])
        
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            sel = np.where(mask == 1)[0]
            
            if len(sel) == 0:
                f1[i], f2[i], f3[i] = 1e6, 1e6, 1e6
                continue
                
            penalty = _size_penalty(len(sel), len(mask))
            
            # Obj 1: Classification Relevance (统一使用 RF Feature Importances)
            f1[i] = -float(np.mean(self.opts['imp_class'][sel])) + penalty
            
            # Obj 2: Redundancy (Minimize Correlation)
            if len(sel) <= 1:
                f2[i] = 0.0 + penalty
            else:
                Xs = self.opts['xt_in'][:, sel]
                Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
                C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
                np.fill_diagonal(C, 0)
                m = C.shape[0]
                f2[i] = float(C.sum() / max(1, m * (m - 1))) + penalty
                
            # Obj 3: Regression Proxy (Minimize negative R2 via LinearRegression)
            model = LinearRegression()
            model.fit(self.opts['xt_in'][:, sel], self.opts['yr_in'])
            r2 = r2_score(self.opts['yr_val'], model.predict(self.opts['xv_in'][:, sel]))
            r2 = max(0.0, float(r2)) 
            f3[i] = -r2 + penalty
            
        out["F"] = np.column_stack([f1, f2, f3])

# ==============================================================
# 2. 定制算子 (MOFS-REPLS & DREA-FS)
# ==============================================================
class REPLSSampling(Sampling):
    """针对 MOFS-REPLS 定制的偏好注入采样器"""
    def _do(self, problem, n_samples, **kwargs):
        D = problem.n_var
        X_init = np.zeros((n_samples, D), dtype=bool)
        weights = problem.opts['imp_combined']
        weights = np.maximum(weights, 1e-8)
        probs = weights / np.sum(weights)
        
        for i in range(n_samples):
            interval = D / (3 * n_samples)
            sel_num = int(1 + np.ceil(interval * (i + 1)))
            sel_num = min(sel_num, D)
            chosen_features = np.random.choice(D, sel_num, replace=False, p=probs)
            X_init[i, chosen_features] = True
            
        return X_init

class MOFS_REPLS(NSGA2):
    """MOFS-REPLS 完整生命周期类"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.me_archive = {} 
        self.nd_archive = []
        self.P = 5 

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        for ind in self.pop:
            x = ind.get("X")
            f_err = ind.get("F")[0] + ind.get("F")[2] 
            num_f = np.sum(x)
            if num_f not in self.me_archive or f_err < self.me_archive[num_f].get("F")[0] + self.me_archive[num_f].get("F")[2]:
                self.me_archive[num_f] = ind
                
        fronts = self.pop.get("rank")
        if fronts is not None:
            nd_indices = np.where(fronts == 0)[0]
            self.nd_archive = self.pop[nd_indices].copy()

    def _infill(self):
        infills = super()._infill()
        if len(self.nd_archive) >= self.P:
            X_inf = infills.get("X")
            for i in range(len(X_inf)):
                ind_float = X_inf[i].astype(float)
                dists = [np.linalg.norm(ind_float - nd.get("X").astype(float)) for nd in self.nd_archive]
                
                p_nearest_indices = np.argsort(dists)[:self.P]
                leader = self.nd_archive[np.random.choice(p_nearest_indices)]
                leader_features = np.where(leader.get("X"))[0]
                
                rand_val = np.random.rand()
                if rand_val < 0.33 and len(leader_features) > 0: 
                    add_f = np.random.choice(leader_features, max(1, len(leader_features)//2), replace=False)
                    X_inf[i, add_f] = True
                elif rand_val < 0.66: 
                    current_features = np.where(X_inf[i])[0]
                    if len(current_features) > 1:
                        del_f = np.random.choice(current_features, max(1, len(current_features)//4), replace=False)
                        X_inf[i, del_f] = False

            infills.set("X", X_inf)
        return infills

class DREAFSSampling(Sampling):
    """针对 DREA-FS 定制的双视角降维采样器"""
    def _do(self, problem, n_samples, **kwargs):
        X_data = problem.opts['xt_in']
        imp_combined = problem.opts['imp_combined']
        n_features = X_data.shape[1]
        
        X_init = np.zeros((n_samples, n_features), dtype=bool)
        half_pop = n_samples // 2
        
        knee_point = np.median(imp_combined[imp_combined > 0]) if np.any(imp_combined > 0) else 0
        important_features = np.where(imp_combined >= knee_point)[0]
        
        for i in range(half_pop):
            if i % 2 == 0:
                mask = np.random.choice([True, False], size=len(important_features))
            else:
                mask = ~mask
            if len(important_features) > 0:
                X_init[i, important_features] = mask
                
        m_groups = max(2, int(5 * np.log2(n_features) if n_features > 0 else 2))
        m_groups = min(m_groups, n_features)
        
        kmeans = KMeans(n_clusters=m_groups, random_state=42, n_init=1)
        labels = kmeans.fit_predict(X_data.T)
        
        for i in range(half_pop, n_samples):
            selected_groups = np.random.choice(m_groups, m_groups // 2, replace=False)
            for g in selected_groups:
                features_in_g = np.where(labels == g)[0]
                if len(features_in_g) > 0:
                    chosen_f = np.random.choice(features_in_g, max(1, len(features_in_g)//2), replace=False)
                    X_init[i, chosen_f] = True
                    
        return X_init

class DREA_FS(NSGA2):
    """DREA-FS 完整生命周期类"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.archive_c = []  
        
    def _infill(self):
        infills = super()._infill()
        if len(self.archive_c) > 0 and self.n_gen % 5 == 0:
            replace_idx = np.random.choice(len(infills), min(5, len(infills)), replace=False)
            X_inf = infills.get("X")
            for idx in replace_idx:
                X_inf[idx] = self.archive_c[np.random.randint(len(self.archive_c))]
            infills.set("X", X_inf)
        return infills

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        F = self.pop.get("F")
        X = self.pop.get("X")
        elite_indices = np.argsort(F[:, 0] + F[:, 2])[:10]
        self.archive_c.extend(X[elite_indices])
        if len(self.archive_c) > 50:
            self.archive_c = self.archive_c[-50:]

# ==============================================================
# 3. Core Evaluation Routine (7 大 MOEAs 并跑)
# ==============================================================
def run_moeas_benchmark_safe(features, class_labels, regression_target, n_runs=10):
    y_class = np.asarray(class_labels)
    y_reg = np.asarray(regression_target)
    feat = normalize(np.asarray(features), norm='l1') 
    
    classifiers = {
        "KNN": KNeighborsClassifier(3), "LDA": LinearDiscriminantAnalysis(),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "SVM": SVC(kernel='linear', random_state=42),
        "RF": RandomForestClassifier(n_estimators=50, random_state=42)
    }

    regressors = {
        "PLS": PLSRegression(n_components=5), "SVR": SVR(kernel='linear'),
        "RF": RandomForestRegressor(n_estimators=50, random_state=42) 
    }

    all_results = []
    
    print("=" * 70)
    print(f"Starting UNIFIED MOEA BENCHMARK (Class + Reg): {n_runs} Runs")
    print("=" * 70)

    for run_idx in range(n_runs):
        print(f"\n>>> Run {run_idx+1}/{n_runs}...")
        current_seed = 42 + run_idx
        set_seed(current_seed)
        
        xt_tr, xt_te, yc_tr, yc_te, yr_tr, yr_te = train_test_split(
            feat, y_class, y_reg, test_size=0.3, stratify=y_class, random_state=current_seed
        )
        
        xt_in, xv_in, yc_in, yc_val, yr_in, yr_val = train_test_split(
            xt_tr, yc_tr, yr_tr, test_size=0.25, stratify=yc_tr, random_state=current_seed
        )
        
        print("  [Pre-compute] Calculating Feature Importances (Random Forest)...")
        rf_c = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1).fit(xt_in, yc_in)
        imp_class = rf_c.feature_importances_
        
        rf_r = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1).fit(xt_in, yr_in)
        imp_reg = rf_r.feature_importances_
        
        imp_combined = (imp_class / max(1e-8, imp_class.max())) + (imp_reg / max(1e-8, imp_reg.max()))
        
        opts = {
            'xt_in': xt_in, 'yc_in': yc_in, 'yr_in': yr_in,
            'xv_in': xv_in, 'yc_val': yc_val, 'yr_val': yr_val,
            'imp_class': imp_class, 'imp_reg': imp_reg, 'imp_combined': imp_combined
        }
        
        problem = DualTaskFeatureSelection(opts)
        
        ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
        pop_size = len(ref_dirs)
        
        crossover = SinglePointCrossover(prob=0.8)
        mutation = BitflipMutation(prob=0.01)
        std_sampling = BinaryRandomSampling()
        
        algorithms = {
            "NSGA-II": NSGA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "NSGA-III": NSGA3(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "SPEA2": SPEA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "RVEA": RVEA(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "MOEA/D": MOEAD(ref_dirs=ref_dirs, n_neighbors=15, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "MOFS-REPLS": MOFS_REPLS(pop_size=pop_size, sampling=REPLSSampling(), crossover=crossover, mutation=mutation),
            "DREA-FS": DREA_FS(pop_size=pop_size, sampling=DREAFSSampling(), crossover=crossover, mutation=mutation)
        }
        
        def get_top_k(scores, k):
            idx = np.argsort(scores)[::-1][:k]
            return idx if len(idx) > 0 else [0]
            
        print("  [Optimizing] Running MOEAs...")
        for algo_name, algo in algorithms.items():
            
            # --- 核心修改：记录运算耗时 ---
            start_time = time.time()
            res = minimize(problem, algo, ('n_gen', 40), seed=current_seed, verbose=False)
            elapsed_time = time.time() - start_time
            
            final_pop = res.pop.get("X").astype(int)
            scores_algo = final_pop.mean(axis=0)
            
            for k in range(1, 16):
                idx = get_top_k(scores_algo, k)
                
                # 分类任务验证
                for clf_name, clf in classifiers.items():
                    clf.fit(xt_tr[:, idx], yc_tr)
                    acc = clf.score(xt_te[:, idx], yc_te)
                    all_results.append({
                        "Run": run_idx+1, "Method": algo_name, "K": k, 
                        "Task": "Classification", "Model": clf_name, "Score": acc,
                        "Time_Seconds": elapsed_time
                    })
                
                # 回归任务验证
                for reg_name, reg in regressors.items():
                    if reg_name == "PLS": reg = PLSRegression(n_components=min(k, 5))
                    reg.fit(xt_tr[:, idx], yr_tr)
                    r2 = r2_score(yr_te, reg.predict(xt_te[:, idx]))
                    all_results.append({
                        "Run": run_idx+1, "Method": algo_name, "K": k, 
                        "Task": "Regression", "Model": reg_name, "Score": r2,
                        "Time_Seconds": elapsed_time
                    })

    # ==============================================================
    # 4. Data Summary & Output
    # ==============================================================
    df = pd.DataFrame(all_results)
    
    def calc_mean_std(x):
        return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"

    print("\n" + "="*80)
    print("             UNIFIED MOEA TABLES (Mean ± Std)")
    print("="*80)
    
    # 分类任务 LDA 提取
    df_lda = df[(df["Task"] == "Classification") & (df["Model"] == "LDA")]
    if not df_lda.empty:
        table_lda = df_lda.pivot_table(index="K", columns="Method", values="Score", aggfunc=calc_mean_std)
        table_lda = table_lda[[m for m in METHOD_ORDER if m in table_lda.columns]]
        table_lda.to_csv("Unified_Class_LDA.csv")
        print("\n[TABLE 1] Classification Accuracy [All 7 MOEAs]:\n", table_lda)

    # 回归任务 RF 提取
    df_rf = df[(df["Task"] == "Regression") & (df["Model"] == "RF")]
    if not df_rf.empty:
        table_rf = df_rf.pivot_table(index="K", columns="Method", values="Score", aggfunc=calc_mean_std)
        table_rf = table_rf[[m for m in METHOD_ORDER if m in table_rf.columns]]
        table_rf.to_csv("Unified_Reg_RF.csv")
        print("\n[TABLE 2] Regression R2 Score [All 7 MOEAs]:\n", table_rf)

    # 运行时间提取
    table_time = df.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    table_time.to_csv("Unified_Execution_Time.csv")
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm:\n", table_time)

    print("\n-> 大一统 7 算法基线对比及运行时间 CSV 文件生成完毕！")

if __name__ == "__main__":
    try:
        print("Loading Data...")
        df = pd.read_excel("C:/Users/zhan0035/Downloads/DATASET.xlsx")
        label = df.iloc[:, 1].to_numpy()
        sugar = df.iloc[:, 2].to_numpy()
        features = df.drop(df.columns[[0, 1, 2]], axis=1).values
        
        run_moeas_benchmark_safe(features, label, sugar, n_runs=30)
        
    except Exception as e:
        print(f"Data loading failed ({e}). Running Safe Mock Benchmark...")
        X = np.random.rand(150, 50); yc = np.random.randint(0, 2, 150); yr = np.random.rand(150) * 10 + 5
        run_moeas_benchmark_safe(X, yc, yr, n_runs=2)

Loading Data...
Starting UNIFIED MOEA BENCHMARK (Class + Reg): 30 Runs

>>> Run 1/30...
  [Pre-compute] Calculating Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 2/30...
  [Pre-compute] Calculating Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 3/30...
  [Pre-compute] Calculating Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 4/30...
  [Pre-compute] Calculating Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 5/30...
  [Pre-compute] Calculating Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 6/30...
  [Pre-compute] Calculating Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 7/30...
  [Pre-compute] Calculating Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 8/30...
  [Pre-compute] Calculating Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...



In [5]:
# ==============================================================
# 纯净版 Baseline 实验：Sugar Dataset (Classification & Regression)
# 综合对比测试：7 种纯正血统的 SOTA MOEAs (原生 PyMoo 实现，绝无魔改)
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random
import time
from scipy import stats

# --- Sklearn ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.metrics import r2_score, accuracy_score
from sklearn.feature_selection import mutual_info_classif  # 引入互信息

# --- Models ---
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, RandomForestRegressor 
from sklearn.svm import SVC, SVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression

# --- Pymoo (v0.6.0+) 原生纯血算法 ---
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.algorithms.moo.rvea import RVEA
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.algorithms.moo.age import AGEMOEA     # Modern SOTA
from pymoo.algorithms.moo.sms import SMSEMOA     # Hypervolume SOTA

from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排列顺序
METHOD_ORDER = ["NSGA-II", "NSGA-III", "RVEA", "SPEA2", "MOEA/D", "AGE-MOEA", "SMS-EMOA"]

# ==============================================================
# 0. Configuration
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

# ==============================================================
# 1. Safe & Fast Fitness Functions (统一目标空间)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.02):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

class DualTaskFeatureSelection(Problem):
    def __init__(self, opts, **kwargs):
        # 3 个目标：1. 最大化分类互信息, 2. 最小化冗余, 3. 最大化回归R2
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1 = np.zeros(X.shape[0])
        f2 = np.zeros(X.shape[0])
        f3 = np.zeros(X.shape[0])
        
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            sel = np.where(mask == 1)[0]
            
            if len(sel) == 0:
                f1[i], f2[i], f3[i] = 1e6, 1e6, 1e6
                continue
                
            penalty = _size_penalty(len(sel), len(mask))
            
            # Obj 1: Classification Relevance (使用 Mutual Information)
            f1[i] = -float(np.mean(self.opts['imp_class'][sel])) + penalty
            
            # Obj 2: Redundancy (Minimize Correlation)
            if len(sel) <= 1:
                f2[i] = 0.0 + penalty
            else:
                Xs = self.opts['xt_in'][:, sel]
                Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
                C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
                np.fill_diagonal(C, 0)
                m = C.shape[0]
                f2[i] = float(C.sum() / max(1, m * (m - 1))) + penalty
                
            # Obj 3: Regression Proxy (Minimize negative R2 via LinearRegression)
            model = LinearRegression()
            model.fit(self.opts['xt_in'][:, sel], self.opts['yr_in'])
            r2 = r2_score(self.opts['yr_val'], model.predict(self.opts['xv_in'][:, sel]))
            r2 = max(0.0, float(r2)) 
            f3[i] = -r2 + penalty
            
        out["F"] = np.column_stack([f1, f2, f3])


# ==============================================================
# 2. Core Evaluation Routine (7 大纯正 MOEAs 并跑)
# ==============================================================
def run_moeas_benchmark_safe(features, class_labels, regression_target, n_runs=10):
    y_class = np.asarray(class_labels)
    y_reg = np.asarray(regression_target)
    feat = normalize(np.asarray(features), norm='l1') 
    
    classifiers = {
        "KNN": KNeighborsClassifier(3), "LDA": LinearDiscriminantAnalysis(),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "SVM": SVC(kernel='linear', random_state=42),
        "RF": RandomForestClassifier(n_estimators=50, random_state=42)
    }

    regressors = {
        "PLS": PLSRegression(n_components=5), "SVR": SVR(kernel='linear'),
        "RF": RandomForestRegressor(n_estimators=50, random_state=42) 
    }

    all_results = []
    
    print("=" * 80)
    print(f"Starting PURE MOEA BENCHMARK (Class + Reg): {n_runs} Runs")
    print("=" * 80)

    for run_idx in range(n_runs):
        print(f"\n>>> Run {run_idx+1}/{n_runs}...")
        current_seed = 42 + run_idx
        set_seed(current_seed)
        
        xt_tr, xt_te, yc_tr, yc_te, yr_tr, yr_te = train_test_split(
            feat, y_class, y_reg, test_size=0.3, stratify=y_class, random_state=current_seed
        )
        
        xt_in, xv_in, yc_in, yc_val, yr_in, yr_val = train_test_split(
            xt_tr, yc_tr, yr_tr, test_size=0.25, stratify=yc_tr, random_state=current_seed
        )
        
        print("  [Pre-compute] Calculating Mutual Information...")
        # 核心修改：统一使用 mutual_info_classif 保持和之前框架的绝对公平对比
        imp_class = mutual_info_classif(xt_in, yc_in, random_state=current_seed)
        
        opts = {
            'xt_in': xt_in, 'yc_in': yc_in, 'yr_in': yr_in,
            'xv_in': xv_in, 'yc_val': yc_val, 'yr_val': yr_val,
            'imp_class': imp_class
        }
        
        problem = DualTaskFeatureSelection(opts)
        
        ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
        pop_size = len(ref_dirs) # 91
        
        crossover = SinglePointCrossover(prob=0.8)
        mutation = BitflipMutation(prob=0.01)
        std_sampling = BinaryRandomSampling()
        
        algorithms = {
            "NSGA-II": NSGA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "NSGA-III": NSGA3(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "SPEA2": SPEA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "RVEA": RVEA(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "MOEA/D": MOEAD(ref_dirs=ref_dirs, n_neighbors=15, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "AGE-MOEA": AGEMOEA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "SMS-EMOA": SMSEMOA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation)
        }
        
        def get_top_k(scores, k):
            idx = np.argsort(scores)[::-1][:k]
            return idx if len(idx) > 0 else [0]
            
        print("  [Optimizing] Running MOEAs...")
        for algo_name, algo in algorithms.items():
            
            start_time = time.time()
            res = minimize(problem, algo, ('n_gen', 40), seed=current_seed, verbose=False)
            elapsed_time = time.time() - start_time
            
            final_pop = res.pop.get("X").astype(int)
            scores_algo = final_pop.mean(axis=0)
            
            for k in range(1, 16):
                idx = get_top_k(scores_algo, k)
                
                # 分类任务验证
                for clf_name, clf in classifiers.items():
                    clf.fit(xt_tr[:, idx], yc_tr)
                    acc = clf.score(xt_te[:, idx], yc_te)
                    all_results.append({
                        "Run": run_idx+1, "Method": algo_name, "K": k, 
                        "Task": "Classification", "Model": clf_name, "Score": acc,
                        "Time_Seconds": elapsed_time
                    })
                
                # 回归任务验证
                for reg_name, reg in regressors.items():
                    if reg_name == "PLS": 
                        reg = PLSRegression(n_components=min(k, 5))
                    reg.fit(xt_tr[:, idx], yr_tr)
                    r2 = r2_score(yr_te, reg.predict(xt_te[:, idx]))
                    all_results.append({
                        "Run": run_idx+1, "Method": algo_name, "K": k, 
                        "Task": "Regression", "Model": reg_name, "Score": r2,
                        "Time_Seconds": elapsed_time
                    })

    # ==============================================================
    # 3. Data Summary & Output
    # ==============================================================
    df = pd.DataFrame(all_results)
    
    def calc_mean_std(x):
        return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"

    print("\n" + "="*80)
    print("             PURE MOEA TABLES (Mean ± Std)")
    print("="*80)
    
    # 分类任务 LDA 提取
    df_lda = df[(df["Task"] == "Classification") & (df["Model"] == "LDA")]
    if not df_lda.empty:
        table_lda = df_lda.pivot_table(index="K", columns="Method", values="Score", aggfunc=calc_mean_std)
        table_lda = table_lda[[m for m in METHOD_ORDER if m in table_lda.columns]]
        table_lda.to_csv("Pure_MOEA_Class_LDA.csv")
        print("\n[TABLE 1] Classification Accuracy (LDA) [All 7 MOEAs]:\n", table_lda)

    # 回归任务 RF 提取
    df_rf = df[(df["Task"] == "Regression") & (df["Model"] == "RF")]
    if not df_rf.empty:
        table_rf = df_rf.pivot_table(index="K", columns="Method", values="Score", aggfunc=calc_mean_std)
        table_rf = table_rf[[m for m in METHOD_ORDER if m in table_rf.columns]]
        table_rf.to_csv("Pure_MOEA_Reg_RF.csv")
        print("\n[TABLE 2] Regression R2 Score (RF) [All 7 MOEAs]:\n", table_rf)

    # 运行时间提取
    table_time = df.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    table_time.to_csv("Pure_MOEA_Execution_Time.csv")
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm:\n", table_time)

    print("\n-> 纯血 7 种 SOTA 算法基线对比及运行时间 CSV 文件生成完毕！")

if __name__ == "__main__":
    try:
        print("Loading Data...")
        df = pd.read_excel("C:/Users/zhan0035/Downloads/DATASET.xlsx")
        label = df.iloc[:, 1].to_numpy()
        sugar = df.iloc[:, 2].to_numpy()
        features = df.drop(df.columns[[0, 1, 2]], axis=1).values
        
        run_moeas_benchmark_safe(features, label, sugar, n_runs=30)
        
    except Exception as e:
        print(f"Data loading failed ({e}). Running Safe Mock Benchmark...")
        X = np.random.rand(150, 50); yc = np.random.randint(0, 2, 150); yr = np.random.rand(150) * 10 + 5
        run_moeas_benchmark_safe(X, yc, yr, n_runs=2)

Loading Data...
Starting PURE MOEA BENCHMARK (Class + Reg): 30 Runs

>>> Run 1/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 2/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 3/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 4/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 5/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 6/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 7/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 8/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 9/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 10/30...
  [Pre-compute] Calcula

In [8]:
# ==============================================================
# 纯净版 Baseline 实验：Sugar Dataset (Classification & Regression)
# 综合对比测试：7 种纯正血统的 SOTA MOEAs (原生 PyMoo 实现，绝无魔改)
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random
import time
from scipy import stats

# --- Sklearn ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.metrics import r2_score, accuracy_score
from sklearn.feature_selection import mutual_info_classif  # 引入互信息

# --- Models ---
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, RandomForestRegressor 
from sklearn.svm import SVC, SVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression

# --- Pymoo (v0.6.0+) 原生纯血算法 ---
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.algorithms.moo.rvea import RVEA
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.algorithms.moo.age import AGEMOEA     # Modern SOTA
from pymoo.algorithms.moo.sms import SMSEMOA     # Hypervolume SOTA

from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排列顺序
METHOD_ORDER = ["AGE-MOEA", "SMS-EMOA"]

# ==============================================================
# 0. Configuration
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

# ==============================================================
# 1. Safe & Fast Fitness Functions (统一目标空间)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.02):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

class DualTaskFeatureSelection(Problem):
    def __init__(self, opts, **kwargs):
        # 3 个目标：1. 最大化分类互信息, 2. 最小化冗余, 3. 最大化回归R2
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1 = np.zeros(X.shape[0])
        f2 = np.zeros(X.shape[0])
        f3 = np.zeros(X.shape[0])
        
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            sel = np.where(mask == 1)[0]
            
            if len(sel) == 0:
                f1[i], f2[i], f3[i] = 1e6, 1e6, 1e6
                continue
                
            penalty = _size_penalty(len(sel), len(mask))
            
            # Obj 1: Classification Relevance (使用 Mutual Information)
            f1[i] = -float(np.mean(self.opts['imp_class'][sel])) + penalty
            
            # Obj 2: Redundancy (Minimize Correlation)
            if len(sel) <= 1:
                f2[i] = 0.0 + penalty
            else:
                Xs = self.opts['xt_in'][:, sel]
                Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
                C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
                np.fill_diagonal(C, 0)
                m = C.shape[0]
                f2[i] = float(C.sum() / max(1, m * (m - 1))) + penalty
                
            # Obj 3: Regression Proxy (Minimize negative R2 via LinearRegression)
            model = LinearRegression()
            model.fit(self.opts['xt_in'][:, sel], self.opts['yr_in'])
            r2 = r2_score(self.opts['yr_val'], model.predict(self.opts['xv_in'][:, sel]))
            r2 = max(0.0, float(r2)) 
            f3[i] = -r2 + penalty
            
        out["F"] = np.column_stack([f1, f2, f3])


# ==============================================================
# 2. Core Evaluation Routine (7 大纯正 MOEAs 并跑)
# ==============================================================
def run_moeas_benchmark_safe(features, class_labels, regression_target, n_runs=10):
    y_class = np.asarray(class_labels)
    y_reg = np.asarray(regression_target)
    feat = normalize(np.asarray(features), norm='l1') 
    
    classifiers = {
        "KNN": KNeighborsClassifier(3), "LDA": LinearDiscriminantAnalysis(),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "SVM": SVC(kernel='linear', random_state=42),
        "RF": RandomForestClassifier(n_estimators=50, random_state=42)
    }

    regressors = {
        "PLS": PLSRegression(n_components=5), "SVR": SVR(kernel='linear'),
        "RF": RandomForestRegressor(n_estimators=50, random_state=42) 
    }

    all_results = []
    
    print("=" * 80)
    print(f"Starting PURE MOEA BENCHMARK (Class + Reg): {n_runs} Runs")
    print("=" * 80)

    for run_idx in range(n_runs):
        print(f"\n>>> Run {run_idx+1}/{n_runs}...")
        current_seed = 42 + run_idx
        set_seed(current_seed)
        
        xt_tr, xt_te, yc_tr, yc_te, yr_tr, yr_te = train_test_split(
            feat, y_class, y_reg, test_size=0.3, stratify=y_class, random_state=current_seed
        )
        
        xt_in, xv_in, yc_in, yc_val, yr_in, yr_val = train_test_split(
            xt_tr, yc_tr, yr_tr, test_size=0.25, stratify=yc_tr, random_state=current_seed
        )
        
        print("  [Pre-compute] Calculating Mutual Information...")
        # 核心修改：统一使用 mutual_info_classif 保持和之前框架的绝对公平对比
        imp_class = mutual_info_classif(xt_in, yc_in, random_state=current_seed)
        
        opts = {
            'xt_in': xt_in, 'yc_in': yc_in, 'yr_in': yr_in,
            'xv_in': xv_in, 'yc_val': yc_val, 'yr_val': yr_val,
            'imp_class': imp_class
        }
        
        problem = DualTaskFeatureSelection(opts)
        
        ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
        pop_size = len(ref_dirs) # 91
        
        crossover = SinglePointCrossover(prob=0.8)
        mutation = BitflipMutation(prob=0.01)
        std_sampling = BinaryRandomSampling()
        
        algorithms = {
            "NSGA-II": NSGA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "NSGA-III": NSGA3(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "SPEA2": SPEA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "RVEA": RVEA(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "MOEA/D": MOEAD(ref_dirs=ref_dirs, n_neighbors=15, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "AGE-MOEA": AGEMOEA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "SMS-EMOA": SMSEMOA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation)
        }
        
        def get_top_k(scores, k):
            idx = np.argsort(scores)[::-1][:k]
            return idx if len(idx) > 0 else [0]
            
        print("  [Optimizing] Running MOEAs...")
        for algo_name, algo in algorithms.items():
            
            start_time = time.time()
            res = minimize(problem, algo, ('n_gen', 40), seed=current_seed, verbose=False)
            elapsed_time = time.time() - start_time
            
            final_pop = res.pop.get("X").astype(int)
            scores_algo = final_pop.mean(axis=0)
            
            for k in range(1, 16):
                idx = get_top_k(scores_algo, k)
                
                # 分类任务验证
                for clf_name, clf in classifiers.items():
                    clf.fit(xt_tr[:, idx], yc_tr)
                    acc = clf.score(xt_te[:, idx], yc_te)
                    all_results.append({
                        "Run": run_idx+1, "Method": algo_name, "K": k, 
                        "Task": "Classification", "Model": clf_name, "Score": acc,
                        "Time_Seconds": elapsed_time
                    })
                
                # 回归任务验证
                for reg_name, reg in regressors.items():
                    if reg_name == "PLS": 
                        reg = PLSRegression(n_components=min(k, 5))
                    reg.fit(xt_tr[:, idx], yr_tr)
                    r2 = r2_score(yr_te, reg.predict(xt_te[:, idx]))
                    all_results.append({
                        "Run": run_idx+1, "Method": algo_name, "K": k, 
                        "Task": "Regression", "Model": reg_name, "Score": r2,
                        "Time_Seconds": elapsed_time
                    })

    # ==============================================================
    # 3. Data Summary & Output
    # ==============================================================
    df = pd.DataFrame(all_results)
    
    def calc_mean_std(x):
        return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"

    print("\n" + "="*80)
    print("             PURE MOEA TABLES (Mean ± Std)")
    print("="*80)
    
    # 分类任务 LDA 提取
    df_lda = df[(df["Task"] == "Classification") & (df["Model"] == "LDA")]
    if not df_lda.empty:
        table_lda = df_lda.pivot_table(index="K", columns="Method", values="Score", aggfunc=calc_mean_std)
        table_lda = table_lda[[m for m in METHOD_ORDER if m in table_lda.columns]]
        table_lda.to_csv("Pure_MOEA_Class_LDA.csv")
        print("\n[TABLE 1] Classification Accuracy (LDA) [All 7 MOEAs]:\n", table_lda)

    # 回归任务 RF 提取
    df_rf = df[(df["Task"] == "Regression") & (df["Model"] == "RF")]
    if not df_rf.empty:
        table_rf = df_rf.pivot_table(index="K", columns="Method", values="Score", aggfunc=calc_mean_std)
        table_rf = table_rf[[m for m in METHOD_ORDER if m in table_rf.columns]]
        table_rf.to_csv("Pure_MOEA_Reg_RF.csv")
        print("\n[TABLE 2] Regression R2 Score (RF) [All 7 MOEAs]:\n", table_rf)

    # 运行时间提取
    table_time = df.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    table_time.to_csv("Pure_MOEA_Execution_Time.csv")
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm:\n", table_time)

    print("\n-> 纯血 7 种 SOTA 算法基线对比及运行时间 CSV 文件生成完毕！")

if __name__ == "__main__":
    try:
        print("Loading Data...")
        df = pd.read_excel("C:/Users/zhan0035/Downloads/DATASET.xlsx")
        label = df.iloc[:, 1].to_numpy()
        sugar = df.iloc[:, 2].to_numpy()
        features = df.drop(df.columns[[0, 1, 2]], axis=1).values
        
        run_moeas_benchmark_safe(features, label, sugar, n_runs=30)
        
    except Exception as e:
        print(f"Data loading failed ({e}). Running Safe Mock Benchmark...")
        X = np.random.rand(150, 50); yc = np.random.randint(0, 2, 150); yr = np.random.rand(150) * 10 + 5
        run_moeas_benchmark_safe(X, yc, yr, n_runs=2)

Loading Data...
Starting PURE MOEA BENCHMARK (Class + Reg): 30 Runs

>>> Run 1/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 2/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 3/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 4/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 5/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 6/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 7/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 8/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 9/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running MOEAs...

>>> Run 10/30...
  [Pre-compute] Calcula

In [6]:
# ==============================================================
# 定制版 Baseline 实验：Sugar Dataset (Classification & Regression)
# 专项测试：仅运行 MOFS-REPLS 和 DREA-FS (包含专版采样器与算子)
# 核心修正：严格对齐互信息 (Mutual Information) 作为特征评价指标
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random
import time
from scipy import stats

# --- Sklearn ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.metrics import r2_score, accuracy_score
from sklearn.cluster import KMeans
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression  # 引入互信息

# --- Models ---
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, RandomForestRegressor 
from sklearn.svm import SVC, SVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression

# --- Pymoo (v0.6.0+) ---
from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排列顺序（仅保留定制算法）
METHOD_ORDER = ["MOFS-REPLS", "DREA-FS"]

# ==============================================================
# 0. Configuration
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

# ==============================================================
# 1. Safe & Fast Fitness Functions (统一目标空间)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.02):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

class DualTaskFeatureSelection(Problem):
    def __init__(self, opts, **kwargs):
        # 3 个目标：1. 最大化分类互信息, 2. 最小化冗余, 3. 最大化回归R2
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1 = np.zeros(X.shape[0])
        f2 = np.zeros(X.shape[0])
        f3 = np.zeros(X.shape[0])
        
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            sel = np.where(mask == 1)[0]
            
            if len(sel) == 0:
                f1[i], f2[i], f3[i] = 1e6, 1e6, 1e6
                continue
                
            penalty = _size_penalty(len(sel), len(mask))
            
            # Obj 1: Classification Relevance (使用 Mutual Information)
            f1[i] = -float(np.mean(self.opts['imp_class'][sel])) + penalty
            
            # Obj 2: Redundancy (Minimize Correlation)
            if len(sel) <= 1:
                f2[i] = 0.0 + penalty
            else:
                Xs = self.opts['xt_in'][:, sel]
                Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
                C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
                np.fill_diagonal(C, 0)
                m = C.shape[0]
                f2[i] = float(C.sum() / max(1, m * (m - 1))) + penalty
                
            # Obj 3: Regression Proxy (Minimize negative R2 via LinearRegression)
            model = LinearRegression()
            model.fit(self.opts['xt_in'][:, sel], self.opts['yr_in'])
            r2 = r2_score(self.opts['yr_val'], model.predict(self.opts['xv_in'][:, sel]))
            r2 = max(0.0, float(r2)) 
            f3[i] = -r2 + penalty
            
        out["F"] = np.column_stack([f1, f2, f3])


# ==============================================================
# 2. 定制算子 (MOFS-REPLS & DREA-FS)
# ==============================================================
class REPLSSampling(Sampling):
    """针对 MOFS-REPLS 定制的偏好注入采样器"""
    def _do(self, problem, n_samples, **kwargs):
        D = problem.n_var
        X_init = np.zeros((n_samples, D), dtype=bool)
        weights = problem.opts['imp_combined']
        weights = np.maximum(weights, 1e-8)
        probs = weights / np.sum(weights)
        
        for i in range(n_samples):
            interval = D / (3 * n_samples)
            sel_num = int(1 + np.ceil(interval * (i + 1)))
            sel_num = min(sel_num, D)
            chosen_features = np.random.choice(D, sel_num, replace=False, p=probs)
            X_init[i, chosen_features] = True
            
        return X_init

class MOFS_REPLS(NSGA2):
    """MOFS-REPLS 完整生命周期类"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.me_archive = {} 
        self.nd_archive = []
        self.P = 5 

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        for ind in self.pop:
            x = ind.get("X")
            # 综合考虑分类错误率(F0)和回归错误率(F2)
            f_err = ind.get("F")[0] + ind.get("F")[2] 
            num_f = np.sum(x)
            if num_f not in self.me_archive or f_err < self.me_archive[num_f].get("F")[0] + self.me_archive[num_f].get("F")[2]:
                self.me_archive[num_f] = ind
                
        fronts = self.pop.get("rank")
        if fronts is not None:
            nd_indices = np.where(fronts == 0)[0]
            self.nd_archive = self.pop[nd_indices].copy()

    def _infill(self):
        infills = super()._infill()
        if len(self.nd_archive) >= self.P:
            X_inf = infills.get("X")
            for i in range(len(X_inf)):
                ind_float = X_inf[i].astype(float)
                dists = [np.linalg.norm(ind_float - nd.get("X").astype(float)) for nd in self.nd_archive]
                
                p_nearest_indices = np.argsort(dists)[:self.P]
                leader = self.nd_archive[np.random.choice(p_nearest_indices)]
                leader_features = np.where(leader.get("X"))[0]
                
                rand_val = np.random.rand()
                if rand_val < 0.33 and len(leader_features) > 0: 
                    add_f = np.random.choice(leader_features, max(1, len(leader_features)//2), replace=False)
                    X_inf[i, add_f] = True
                elif rand_val < 0.66: 
                    current_features = np.where(X_inf[i])[0]
                    if len(current_features) > 1:
                        del_f = np.random.choice(current_features, max(1, len(current_features)//4), replace=False)
                        X_inf[i, del_f] = False

            infills.set("X", X_inf)
        return infills

class DREAFSSampling(Sampling):
    """针对 DREA-FS 定制的双视角降维采样器"""
    def _do(self, problem, n_samples, **kwargs):
        X_data = problem.opts['xt_in']
        imp_combined = problem.opts['imp_combined']
        n_features = X_data.shape[1]
        
        X_init = np.zeros((n_samples, n_features), dtype=bool)
        half_pop = n_samples // 2
        
        knee_point = np.median(imp_combined[imp_combined > 0]) if np.any(imp_combined > 0) else 0
        important_features = np.where(imp_combined >= knee_point)[0]
        
        for i in range(half_pop):
            if i % 2 == 0:
                mask = np.random.choice([True, False], size=len(important_features))
            else:
                mask = ~mask
            if len(important_features) > 0:
                X_init[i, important_features] = mask
                
        m_groups = max(2, int(5 * np.log2(n_features) if n_features > 0 else 2))
        m_groups = min(m_groups, n_features)
        
        kmeans = KMeans(n_clusters=m_groups, random_state=42, n_init=1)
        labels = kmeans.fit_predict(X_data.T)
        
        for i in range(half_pop, n_samples):
            selected_groups = np.random.choice(m_groups, m_groups // 2, replace=False)
            for g in selected_groups:
                features_in_g = np.where(labels == g)[0]
                if len(features_in_g) > 0:
                    chosen_f = np.random.choice(features_in_g, max(1, len(features_in_g)//2), replace=False)
                    X_init[i, chosen_f] = True
                    
        return X_init

class DREA_FS(NSGA2):
    """DREA-FS 完整生命周期类"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.archive_c = []  
        
    def _infill(self):
        infills = super()._infill()
        if len(self.archive_c) > 0 and self.n_gen % 5 == 0:
            replace_idx = np.random.choice(len(infills), min(5, len(infills)), replace=False)
            X_inf = infills.get("X")
            for idx in replace_idx:
                X_inf[idx] = self.archive_c[np.random.randint(len(self.archive_c))]
            infills.set("X", X_inf)
        return infills

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        F = self.pop.get("F")
        X = self.pop.get("X")
        # 精英归档综合考虑分类和回归任务
        elite_indices = np.argsort(F[:, 0] + F[:, 2])[:10]
        self.archive_c.extend(X[elite_indices])
        if len(self.archive_c) > 50:
            self.archive_c = self.archive_c[-50:]


# ==============================================================
# 3. Core Evaluation Routine (仅运行两个定制算法)
# ==============================================================
def run_moeas_benchmark_safe(features, class_labels, regression_target, n_runs=10):
    y_class = np.asarray(class_labels)
    y_reg = np.asarray(regression_target)
    feat = normalize(np.asarray(features), norm='l1') 
    
    classifiers = {
        "KNN": KNeighborsClassifier(3), "LDA": LinearDiscriminantAnalysis(),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "SVM": SVC(kernel='linear', random_state=42),
        "RF": RandomForestClassifier(n_estimators=50, random_state=42)
    }

    regressors = {
        "PLS": PLSRegression(n_components=5), "SVR": SVR(kernel='linear'),
        "RF": RandomForestRegressor(n_estimators=50, random_state=42) 
    }

    all_results = []
    
    print("=" * 80)
    print(f"Starting CUSTOM MOEAs BENCHMARK (Class + Reg): {n_runs} Runs")
    print("=" * 80)

    for run_idx in range(n_runs):
        print(f"\n>>> Run {run_idx+1}/{n_runs}...")
        current_seed = 42 + run_idx
        set_seed(current_seed)
        
        xt_tr, xt_te, yc_tr, yc_te, yr_tr, yr_te = train_test_split(
            feat, y_class, y_reg, test_size=0.3, stratify=y_class, random_state=current_seed
        )
        
        xt_in, xv_in, yc_in, yc_val, yr_in, yr_val = train_test_split(
            xt_tr, yc_tr, yr_tr, test_size=0.25, stratify=yc_tr, random_state=current_seed
        )
        
        print("  [Pre-compute] Calculating Mutual Information...")
        # 核心修正：使用互信息替代随机森林计算特征重要性
        imp_class = mutual_info_classif(xt_in, yc_in, random_state=current_seed)
        imp_reg = mutual_info_regression(xt_in, yr_in, random_state=current_seed)
        
        # 归一化并融合，供定制采样器使用
        imp_combined = (imp_class / max(1e-8, imp_class.max())) + (imp_reg / max(1e-8, imp_reg.max()))
        
        opts = {
            'xt_in': xt_in, 'yc_in': yc_in, 'yr_in': yr_in,
            'xv_in': xv_in, 'yc_val': yc_val, 'yr_val': yr_val,
            'imp_class': imp_class, 'imp_combined': imp_combined
        }
        
        problem = DualTaskFeatureSelection(opts)
        
        ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
        pop_size = len(ref_dirs) # 91
        
        crossover = SinglePointCrossover(prob=0.8)
        mutation = BitflipMutation(prob=0.01)
        
        # 仅加载 2 种定制算法
        algorithms = {
            "MOFS-REPLS": MOFS_REPLS(pop_size=pop_size, sampling=REPLSSampling(), crossover=crossover, mutation=mutation),
            "DREA-FS": DREA_FS(pop_size=pop_size, sampling=DREAFSSampling(), crossover=crossover, mutation=mutation)
        }
        
        def get_top_k(scores, k):
            idx = np.argsort(scores)[::-1][:k]
            return idx if len(idx) > 0 else [0]
            
        print("  [Optimizing] Running Custom MOEAs...")
        for algo_name, algo in algorithms.items():
            
            start_time = time.time()
            res = minimize(problem, algo, ('n_gen', 40), seed=current_seed, verbose=False)
            elapsed_time = time.time() - start_time
            
            final_pop = res.pop.get("X").astype(int)
            scores_algo = final_pop.mean(axis=0)
            
            for k in range(1, 16):
                idx = get_top_k(scores_algo, k)
                
                # 分类任务验证
                for clf_name, clf in classifiers.items():
                    clf.fit(xt_tr[:, idx], yc_tr)
                    acc = clf.score(xt_te[:, idx], yc_te)
                    all_results.append({
                        "Run": run_idx+1, "Method": algo_name, "K": k, 
                        "Task": "Classification", "Model": clf_name, "Score": acc,
                        "Time_Seconds": elapsed_time
                    })
                
                # 回归任务验证
                for reg_name, reg in regressors.items():
                    if reg_name == "PLS": 
                        reg = PLSRegression(n_components=min(k, 5))
                    reg.fit(xt_tr[:, idx], yr_tr)
                    r2 = r2_score(yr_te, reg.predict(xt_te[:, idx]))
                    all_results.append({
                        "Run": run_idx+1, "Method": algo_name, "K": k, 
                        "Task": "Regression", "Model": reg_name, "Score": r2,
                        "Time_Seconds": elapsed_time
                    })

    # ==============================================================
    # 4. Data Summary & Output
    # ==============================================================
    df = pd.DataFrame(all_results)
    
    def calc_mean_std(x):
        return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"

    print("\n" + "="*80)
    print("             CUSTOM MOEA TABLES (Mean ± Std)")
    print("="*80)
    
    # 分类任务 LDA 提取
    df_lda = df[(df["Task"] == "Classification") & (df["Model"] == "LDA")]
    if not df_lda.empty:
        table_lda = df_lda.pivot_table(index="K", columns="Method", values="Score", aggfunc=calc_mean_std)
        table_lda = table_lda[[m for m in METHOD_ORDER if m in table_lda.columns]]
        table_lda.to_csv("Custom_MOEAs_Class_LDA.csv")
        print("\n[TABLE 1] Classification Accuracy (LDA):\n", table_lda)

    # 回归任务 RF 提取
    df_rf = df[(df["Task"] == "Regression") & (df["Model"] == "RF")]
    if not df_rf.empty:
        table_rf = df_rf.pivot_table(index="K", columns="Method", values="Score", aggfunc=calc_mean_std)
        table_rf = table_rf[[m for m in METHOD_ORDER if m in table_rf.columns]]
        table_rf.to_csv("Custom_MOEAs_Reg_RF.csv")
        print("\n[TABLE 2] Regression R2 Score (RF):\n", table_rf)

    # 运行时间提取
    table_time = df.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    table_time.to_csv("Custom_MOEAs_Execution_Time.csv")
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm:\n", table_time)

    print("\n-> 定制算法 MOFS-REPLS 与 DREA-FS 运行完毕！CSV 文件已生成。")

if __name__ == "__main__":
    try:
        print("Loading Data...")
        df = pd.read_excel("C:/Users/zhan0035/Downloads/DATASET.xlsx")
        label = df.iloc[:, 1].to_numpy()
        sugar = df.iloc[:, 2].to_numpy()
        features = df.drop(df.columns[[0, 1, 2]], axis=1).values
        
        run_moeas_benchmark_safe(features, label, sugar, n_runs=30)
        
    except Exception as e:
        print(f"Data loading failed ({e}). Running Safe Mock Benchmark...")
        X = np.random.rand(150, 50); yc = np.random.randint(0, 2, 150); yr = np.random.rand(150) * 10 + 5
        run_moeas_benchmark_safe(X, yc, yr, n_runs=2)

Loading Data...
Starting CUSTOM MOEAs BENCHMARK (Class + Reg): 30 Runs

>>> Run 1/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 2/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 3/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 4/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 5/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 6/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 7/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 8/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 9/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing]